In [1]:
# ######### used this part for fixing problems running on ARC #

import os


os.environ['HF_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_HUB_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['XDG_CACHE_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['NB_USER'] = 'ishtiahmed'#'ishtiaqueahmedk'
os.environ['TRANSFORMERS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_DATASETS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'




In [2]:
import json
import os
import re
import copy

from typing import Dict, Any, List

from openai import OpenAI
import argparse

In [ ]:
# Modify OpenAI's API key and API base to use the server.
openai_api_key = "xxxxxxx"
openai_api_base = "https://llm-api.arc.vt.edu/api/v1"

client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )

models = client.models.list()
model = models.data[2].id
# print(models.data[0].id)
print(model)

gpt-oss-120b


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Briefly answer: What is the capital of Virginia?"},
]


chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
    )
response_text = chat_completion.choices[0].message.content
print(f"**Model ({model}) Response:**\n{response_text}")


**Model (gpt-oss-120b) Response:**
Richmond.


In [5]:
def call_arc_llm(prompt: str):
    messages = [
    {"role": "system", "content": "You are a helpful assistant who closely follows rules and generates JSON."},
    {"role": "user", "content": prompt},
]


    chat_completion = client.chat.completions.create(
            messages=messages,
            model=model,
            temperature=0.7
        )
    response_text =  chat_completion.choices[0].message.content    
    return response_text

def call_gpt_api(prompt: str):

    # # For OpenAI:
    import openai
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content


def call_claude_api(prompt: str):

    # # For Anthropic Claude:
    import anthropic
    client = anthropic.Anthropic(api_key="your-api-key")
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text
    

def call_llm(prompt: str) -> str:
    """
    Call your LLM model here.
    Replace this function with your actual LLM call.
        
    # # For OpenAI:
    # outputs =  call_gpt_api(prompt)
    
    # # For Anthropic Claude:
    # outputs =  call_claude_api(prompt)
    """
    # PLACEHOLDER - Replace with your actual LLM call

#     input_text = prompt
# #         print(input_text)
#     input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")
    
#     outputs = model.generate(input_ids)
#     outputs = tokenizer.decode(outputs[0][1:-1])
    
#     print("outputs: ", outputs)

    # raise NotImplementedError("Replace this with your LLM call")


    outputs =  call_arc_llm(prompt)
    # print(f"Response:**\n{outputs}")

    return outputs




In [9]:

def remove_similarity_scores(question: Dict[str, Any]) -> Dict[str, Any]:
    """Remove similarity_scores from a question entry."""
    clean_question = question.copy()
    if 'similarity_scores' in clean_question:
        del clean_question['similarity_scores']
    return clean_question

def create_prompt(question: Dict[str, Any]) -> str:
    """Create the full prompt with the question JSON."""
    clean_question = remove_similarity_scores(question)
    input_json = json.dumps(clean_question, indent=2)
    return PROMPT_TEMPLATE.replace("{input_json}", input_json)



---
# Task 3 generation code -- Negation
---

In [6]:


PROMPT_TEMPLATE = """ You are given a single multiple-choice visual question answering item in JSON format. It contains:
"question"
"options": an object with keys "A"–"D" and text descriptions
"correct_answer": a key "A", "B", "C", or "D"
Example Input:
    "question": "Which description matches best with the image?",
    "options": {
      "A": "This image shows a Least Auklet object which has a small black and white body, a black head and back, a white belly, a black bill, a white eye-ring, black wings with white spots, and black legs and feet.",
      "B": "This image shows a Rhinoceros Auklet object which has a black or dark grey body, a large orange bill, a white face with a black stripe behind the eye, and is often seen near water, catching fish.",
      "C": "This image shows a Crested Auklet object which has a small, dark body, a bright orange bill, distinctive white facial markings, and a prominent crest of feathers on its head.",
      "D": "This image shows a Parakeet Auklet object which has a small, stocky body, black head and back, white underparts, an orange bill, and a distinctive white eyebrow stripe, distinguishing it from similar seabirds."
    },
    "correct_answer": "C",
Your task is to rewrite all answer options into the SAME negated style:
  “This image shows a dog, which does not have …” and then listing some visual-only attributes.
Follow these rules for Creating Negated Answer Choices (All Options in Negated Form)
Applies to all answer choices (A–D):
1. Remove all class names from every answer choice.
2. Rewrite every answer choice to begin with: “This image shows a dog which does not have…”
For the correct answer choice:
3. Only include visual attributes that the object in the image truly does not have (true negatives).
4. These true-negative attributes should be taken from:
* Attributes mentioned in the original incorrect answer choices, or
* Clear visual opposites of the object’s real features.
5. You may change the colors of these attributes when negating them (e.g., “does not have a green bill”), as long as the statement remains factually true.
6. Do not negate any feature that the object actually has (no false negatives).
7. Include around 3 to 5 true-negative attributes in the correct answer.
For each incorrect answer choice:
8. Identify the visual attributes that the object actually has from the original correct answer choice.
9. In each incorrect answer choice, negate any two of these true attributes (creating false negatives). Example: if the object has a tentacle, say “does not have a tentacle.”
10. Add two more incorrect visual attributes if needed, but they must remain false with respect to the correct answer choice. You may get these features from the original incorrect answer choice.
11. You may randomly change the colors of these additional visual attributes while negating them (e.g., “does not have a blue fin” instead of its real orange fin). Do not change the colors of the negated real attributes from Step 09. Random colors may only be applied to the extra fake attributes you add, not to the negated real ones.
Additional formatting guidelines:
* Only use visual attributes (color, shape, size, markings, body parts).
* Do not include behavior, habitat, or non-visual information.
* All answer choices must follow the same negated style.
* Ensure that two consecutive incorrect choices do not have more than one common false negatives added in step 9.
Example Negated Version Output (for a non-food item):
    "question": "Which description matches best with the image?",
    "options": {
      "A": "This image shows a dog which does not have a small dark body, a bright orange bill, a black head and back, a red belly, a green bill, a white eye-ring, black wings with white spots, and black legs and feet.",
      "B": "This image shows a dog which does not have a bright orange bill, distinctive white facial markings, a black or dark grey body, a large green bill, a white face with a black stripe behind the eye.",
      "C": "This image shows a dog which does not have black wings with white spots, a bright green bill, distinctive yellow facial markings, a distinctive white eyebrow stripe and a white eye-ring.",
      "D": "This image shows a dog which does not have distinctive white facial markings, a prominent crest of feathers on its head, orange head and back, white underparts, an orange bill, and a distinctive white eyebrow stripe."
    },
    "correct_answer": "C",

Now apply this to this MCQ Entry:

{input_json}
"""

In [7]:
import time

def parse_llm_response(response: str) -> Dict[str, Any]:
    """Parse the LLM response to extract JSON."""
    # Try to extract JSON from response
    # Some LLMs might wrap it in markdown code blocks
    response = response.strip()
    
    # Remove markdown code blocks if present
    if response.startswith("```json"):
        response = response[7:]
    elif response.startswith("```"):
        response = response[3:]
    
    if response.endswith("```"):
        response = response[:-3]
    
    response = response.strip()
    
    # Parse JSON
    return json.loads(response)

def process_all_questions(input_filepath: str, output_filepath: str):
    """
    Main function to process all questions.
    
    Args:
        input_filepath: Path to input JSON file
        output_filepath: Path to output JSON file
    """
    # Load original questions
    print(f"Loading questions from {input_filepath}...")
    # questions = load_questions(input_filepath)

    # with open(input_filepath, 'r', encoding='utf-8') as f: # claude
    #     questions = json.load(f)

    with open(input_filepath, 'r') as f:
        questions = json.load(f)


    print(f"Loaded {len(questions)} questions")
    
    # Process each question
    negated_questions = []
    failed_questions = []
    failed_indices = []
    all_negated_questions = []
    
    for i, question in enumerate(questions):
        # print(f"\nProcessing question {i+1}/{len(questions)}: {question['mcq_id']} ({question['difficulty']})")
        
        try:
            # Create prompt
            prompt = create_prompt(question)
            
            # Call LLM
            # print("  Calling LLM...")
            response = call_llm(prompt)
            
            # Parse response
            # print("  Parsing response...")
            negated_question = parse_llm_response(response)

            all_negated_questions.append(negated_question)
            
            # Validate that key fields are preserved
            if "mcq_id" in negated_question:
                assert negated_question['mcq_id'] == question['mcq_id']
            else:
                negated_question['mcq_id'] = question['mcq_id']
            if "difficulty" in negated_question:
                assert negated_question['difficulty'] == question['difficulty']
            else:
                negated_question['difficulty'] = question['difficulty']

            assert negated_question['correct_answer'] == question['correct_answer']
                

            
            negated_questions.append(negated_question)
            # print("  ✓ Success")
            
        except Exception as e:
            print(f"  ✗ Error processing question: {e}")
            print(i)
            print(f"  Skipping this question...")
            failed_questions.append(question)
            failed_indices.append(i)
            continue
        if i%20==0:
            print(i)
    
    # Save results --- Claude
    print(f"\nSaving {len(negated_questions)} negated questions to {output_filepath}...")
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(negated_questions, f, indent=2, ensure_ascii=False)

    # with open(output_filename, 'w') as f:
    #     json.dump(new_data, f, indent=4)
    # print(f"\n✅ Successfully saved the modified data to {output_filename}")
    
    print("Done!")
    print(f"Successfully processed: {len(negated_questions)}/{len(questions)}")
    return failed_questions, failed_indices, all_negated_questions


# Run Code

In [10]:



# Example usage
# Configure file paths


# 1. Define the filename




INPUT_FILE = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_dogs_with_class_descriptions.json"
OUTPUT_FILE = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/task_3_negated_questions_dog.json"

# Process all questions
failed_questions, failed_indices, all_negated_questions = process_all_questions(INPUT_FILE, OUTPUT_FILE)




Loading questions from /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_dogs_with_class_descriptions.json...
Loaded 240 questions
0
20
40
60
80
100
120
140
160
180
200
220

Saving 240 negated questions to /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/task_3_negated_questions_dog.json...
Done!
Successfully processed: 240/240


In [11]:
print(f"len(failed_questions): {len(failed_questions)}")
print(f"len(failed_indices): {len(failed_questions)}")

len(failed_questions): 0
len(failed_indices): 0
